# Generate a single abstract (debug notebook)

Mirrors the logic of `generate_test_csv_gpt.py`, but only processes **one** `is_test=True` node.
Prints the neighbour publications (id + abstract) used as input, and the original abstract being reconstructed, before calling the OpenAI API.

In [ ]:
import os
import time

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI, RateLimitError

load_dotenv()

In [ ]:
# ---------------------------------------------------------------------------
# Config (replaces argparse CLI args from generate_test_csv_gpt.py)
# ---------------------------------------------------------------------------
nodes_path = "../ogbn_arxiv_dataset/nodes_subset.csv"
edges_path = "../ogbn_arxiv_dataset/edges_subset.csv"
model = "gpt-5.4-mini"
max_new_tokens = 256
temperature = 0.35
top_p = 0.90

In [ ]:
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

In [ ]:
print("Loading data …")
nodes_df = pd.read_csv(nodes_path)
edges_df = pd.read_csv(edges_path)

# Only test nodes are evaluated; all nodes (incl. neighbours) provide text
test_nodes = nodes_df[nodes_df["is_test"] == 1].reset_index(drop=True)
node_lookup = nodes_df.set_index("node_id")  # fast neighbour text lookup

# Pre-build neighbour list per src node
neighbours_of = edges_df.groupby("src_node_id")["dst_node_id"].apply(list).to_dict()

In [ ]:
# ---------------------------------------------------------------------------
# Helper: build prompt
# ---------------------------------------------------------------------------
_VICUNA_SYSTEM = (
    "A chat between a curious user and an artificial intelligence assistant. "
    "The assistant gives helpful, detailed, and polite answers to the user's questions."
)

_USER_INSTRUCTION = (
    "You are a scientific writing assistant. "
    "Based on the source publications listed below, write a concise abstract "
    "for a new application paper that builds on and combines these works. "
    "Output ONLY the abstract text, without any heading or explanation."
)


def build_prompt(neighbour_ids: list[int]) -> tuple[str, str]:
    sources = []
    for nid in neighbour_ids:
        if nid not in node_lookup.index:
            continue
        row = node_lookup.loc[nid]
        title    = str(row.get("title", "")).strip()
        abstract = str(row.get("abstract", "")).strip()
        sources.append(f"- Title: {title}\n  Abstract: {abstract}")

    sources_text = "\n\n".join(sources) if sources else "(no neighbouring sources available)"
    user_message = (
        f"{_USER_INSTRUCTION}\n\n"
        f"Source publications:\n{sources_text}\n\n"
        f"Generated abstract:"
    )
    return _VICUNA_SYSTEM, user_message

In [ ]:
# ---------------------------------------------------------------------------
# Pick ONE test node and inspect its inputs before generating
# ---------------------------------------------------------------------------
row = test_nodes.iloc[0]

node_id            = int(row["node_id"])
mag_id             = row["mag_id"]
original_abstract  = str(row.get("abstract", "")).strip()
neighbour_ids      = neighbours_of.get(node_id, [])

print(f"node_id={node_id}  mag_id={mag_id}  neighbours={len(neighbour_ids)}")
print()

print("=== Input (neighbour) abstracts ===")
for nid in neighbour_ids:
    if nid not in node_lookup.index:
        continue
    n_row = node_lookup.loc[nid]
    n_mag_id   = n_row.get("mag_id", "")
    n_title    = str(n_row.get("title", "")).strip()
    n_abstract = str(n_row.get("abstract", "")).strip()
    print(f"--- node_id={nid}  mag_id={n_mag_id} ---")
    print(f"Title: {n_title}")
    print(f"Abstract: {n_abstract}")
    print()

print("=== Original abstract to recreate ===")
print(f"node_id={node_id}  mag_id={mag_id}")
print(original_abstract)

In [ ]:
# ---------------------------------------------------------------------------
# Generate (same retry logic as generate_test_csv_gpt.py)
# ---------------------------------------------------------------------------
system_msg, user_msg = build_prompt(neighbour_ids)

generated = None
for attempt in range(6):
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_msg},
                {"role": "user",   "content": user_msg},
            ],
            max_completion_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
        )
        generated = response.choices[0].message.content.strip()
        break
    except RateLimitError:
        wait = 2 ** attempt
        print(f"  Rate limit — retrying in {wait}s …", flush=True)
        time.sleep(wait)
    except Exception as e:
        generated = f"ERROR: {type(e).__name__}: {e}"
        break
if generated is None:
    generated = "ERROR: max retries exceeded"

print("=== Generated abstract ===")
print(generated)